<a href="https://colab.research.google.com/github/durgeshptl/Slip-off-Tongue/blob/version3/demo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Fri Jan 23 19:19:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.listdir

<function posix.listdir(path=None)>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

DATA_PATH = "/content/drive/MyDrive/BERT/train.csv"

import pandas as pd
df = pd.read_csv(DATA_PATH)
print(df.shape)

(257413, 8)


In [5]:
LABELS = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

# Ensure text is string + light cleaning (safe for transformers)
df["comment_text"] = df["comment_text"].astype(str)
df["comment_text"] = df["comment_text"].str.replace(r"http\S+|www\.\S+", " ", regex=True)
df["comment_text"] = df["comment_text"].str.replace(r"\s+", " ", regex=True).str.strip()

# Keep only required columns and drop nulls
df = df[["comment_text"] + LABELS].dropna()

# Train/Validation split
from sklearn.model_selection import train_test_split
import numpy as np

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

y_train = train_df[LABELS].values.astype(np.float32)
y_val   = val_df[LABELS].values.astype(np.float32)

print("✅ Train:", train_df.shape, " Val:", val_df.shape)
print("Label totals (train):")
print(pd.DataFrame(y_train, columns=LABELS).sum())

✅ Train: (205930, 7)  Val: (51483, 7)
Label totals (train):
toxic            29835.0
severe_toxic      1303.0
obscene           6744.0
threat             402.0
insult           23860.0
identity_hate    14902.0
dtype: float32


In [6]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128  # good for T4 speed

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_ds = ToxicDataset(train_df["comment_text"].tolist(), y_train)
val_ds   = ToxicDataset(val_df["comment_text"].tolist(), y_val)

print("✅ Dataset ready:", len(train_ds), len(val_ds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

✅ Dataset ready: 205930 51483


In [8]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed

set_seed(42)

# ✅ Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type="multi_label_classification"
)

# ✅ Metrics (Macro F1)
THRESHOLD = 0.5
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))      # sigmoid
    preds = (probs >= THRESHOLD).astype(int)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0)
    }

# ✅ Training args (T4-friendly)
OUT_DIR = "xlmr_out"
EPOCHS = 1
BATCH_TRAIN = 16
BATCH_EVAL  = 16
LR = 2e-5

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,          # ✅ big speed boost on T4
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1
1,0.072600,0.068295,0.604156,0.781408


TrainOutput(global_step=12871, training_loss=0.0886187987230882, metrics={'train_runtime': 2344.4196, 'train_samples_per_second': 87.838, 'train_steps_per_second': 5.49, 'total_flos': 1.354610139001344e+16, 'train_loss': 0.0886187987230882, 'epoch': 1.0})

In [9]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Get predictions
pred_output = trainer.predict(val_ds)

logits = pred_output.predictions
y_true = pred_output.label_ids

# Convert logits → probabilities → binary predictions
probs = 1 / (1 + np.exp(-logits))
y_pred = (probs >= 0.5).astype(int)

print("MICRO F1:", f1_score(y_true, y_pred, average="micro"))
print("MACRO F1:", f1_score(y_true, y_pred, average="macro"))
print("WEIGHTED F1:", f1_score(y_true, y_pred, average="weighted"))

# Per-label report
print("\nClassification Report (per label):\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=LABELS,
    zero_division=0
))

MICRO F1: 0.7814078282828283
MACRO F1: 0.6041557919621615
WEIGHTED F1: 0.7782432441208696

Classification Report (per label):

               precision    recall  f1-score   support

        toxic       0.83      0.81      0.82      7522
 severe_toxic       0.54      0.48      0.51       292
      obscene       0.80      0.85      0.82      1705
       threat       0.00      0.00      0.00        76
       insult       0.81      0.79      0.80      6080
identity_hate       0.73      0.62      0.67      3773

    micro avg       0.80      0.76      0.78     19448
    macro avg       0.62      0.59      0.60     19448
 weighted avg       0.79      0.76      0.78     19448
  samples avg       0.11      0.11      0.11     19448



In [10]:
DRIVE_SAVE_DIR = "/content/drive/MyDrive/BERT"  # change folder name if you want

trainer.model.save_pretrained(DRIVE_SAVE_DIR)
tokenizer.save_pretrained(DRIVE_SAVE_DIR)

print("✅ Saved to Drive:", DRIVE_SAVE_DIR)

✅ Saved to Drive: /content/drive/MyDrive/BERT


In [11]:
import numpy as np
import torch

LABELS = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

def predict_percent(text):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits[0].cpu().numpy()

    probs = 1 / (1 + np.exp(-logits))
    return dict(zip(LABELS, (probs * 100).round(2)))


# 🔹 TEST SAMPLES
test_texts = [
    "I respectfully disagree with your opinion.",
    "You are a useless idiot.",
    "I will kill you if you come here again.",
    "These people are disgusting and should not exist.",
    "Go f*** yourself, you piece of sh*t.",
    "यह एक सामान्य चर्चा है, कोई समस्या नहीं है madarchod",
    "तू बहुत बेवकूफ है और कुछ नहीं जानता।",
    "अगर तू दुबारा दिखा तो जान से मार दूँगा।",
    "tum bilkul bakwas karte ho bhosadi vale"
]

# 🔹 RUN TESTS
for text in test_texts:
    print("\nTEXT:", text)
    for k, v in predict_percent(text).items():
        print(f"{k:15s}: {v:6.2f}%")


TEXT: I respectfully disagree with your opinion.
toxic          :   0.14%
severe_toxic   :   0.02%
obscene        :   0.07%
threat         :   0.04%
insult         :   0.06%
identity_hate  :   0.06%

TEXT: You are a useless idiot.
toxic          :  99.10%
severe_toxic   :   9.38%
obscene        :  66.23%
threat         :   2.07%
insult         :  92.41%
identity_hate  :  20.93%

TEXT: I will kill you if you come here again.
toxic          :  84.42%
severe_toxic   :   2.85%
obscene        :  61.75%
threat         :   2.73%
insult         :  25.28%
identity_hate  :   3.37%

TEXT: These people are disgusting and should not exist.
toxic          :  46.27%
severe_toxic   :   0.05%
obscene        :   0.92%
threat         :   0.12%
insult         :  11.22%
identity_hate  :   2.84%

TEXT: Go f*** yourself, you piece of sh*t.
toxic          :  99.12%
severe_toxic   :  23.55%
obscene        :  95.22%
threat         :   4.76%
insult         :  86.01%
identity_hate  :  13.51%

TEXT: यह एक सामान्य